In [0]:
%sql
USE CATALOG adb_dream_analytics_prod;
CREATE SCHEMA IF NOT EXISTS gold MANAGED LOCATION 'abfss://gold@adlsgen2dreamprod.dfs.core.windows.net/dream_app/';
USE SCHEMA gold;

In [0]:
%sql
CREATE TABLE IF NOT EXISTS gold.dim_customers(
    customer_sk BIGINT GENERATED ALWAYS AS IDENTITY,
    customer_id INT,
    first_name STRING,
    last_name STRING,
    full_name STRING,
    email STRING,
    phone_number STRING,
    city STRING,
    signup_date DATE,
    last_updated_timestamp TIMESTAMP,
    scd2_hash STRING,
    start_date DATE,
    end_date DATE,
    is_current BOOLEAN,
    _processed_at TIMESTAMP
) USING DELTA;

CREATE OR REPLACE TEMPORARY VIEW v_customers_source AS
SELECT
    customer_id,
    first_name,
    last_name,
    concat_ws(' ', first_name, last_name) AS full_name,
    email,
    phone_number,
    city,
    signup_date,
    try_cast(last_updated_timestamp AS TIMESTAMP) AS last_updated_timestamp,
    md5(upper(trim(city))) AS source_hash,
    current_timestamp() as _processed_at
FROM adb_dream_analytics_prod.silver.customers;

MERGE INTO dim_customers AS target
USING (
    SELECT 
        s.customer_id AS merge_key,
        s.*
    FROM v_customers_source s
    JOIN dim_customers t ON s.customer_id = t.customer_id
    WHERE t.is_current = true AND (
        s.source_hash <> t.scd2_hash OR
        s.first_name IS DISTINCT FROM t.first_name OR
        s.last_name IS DISTINCT FROM t.last_name OR
        s.email IS DISTINCT FROM t.email OR
        s.phone_number IS DISTINCT FROM t.phone_number OR
        s.last_updated_timestamp IS DISTINCT FROM t.last_updated_timestamp
    )

    UNION ALL

    SELECT
        NULL AS merge_key,
        s.*
    FROM v_customers_source s
    LEFT JOIN dim_customers t ON s.customer_id = t.customer_id
        AND t.is_current = true
        AND t.scd2_hash = s.source_hash
    WHERE t.customer_id IS NULL
) AS source

ON target.customer_id = source.merge_key
    AND target.is_current = true

WHEN MATCHED AND target.scd2_hash <> source.source_hash THEN
    UPDATE SET
        target.is_current = false,
        target.end_date = current_timestamp()

WHEN MATCHED AND target.scd2_hash = source.source_hash AND(
    target.first_name IS DISTINCT FROM source.first_name OR
    target.last_name IS DISTINCT FROM source.last_name OR
    target.email  IS DISTINCT FROM source.email OR
    target.phone_number IS DISTINCT FROM source.phone_number
) THEN
    UPDATE SET
        target.first_name = source.first_name,
        target.last_name = source.last_name,
        target.full_name = source.full_name,
        target.email = source.email,
        target.phone_number = source.phone_number,
        target.last_updated_timestamp = source.last_updated_timestamp,
        target._processed_at = source._processed_at

WHEN NOT MATCHED THEN
    INSERT (
        customer_id, first_name, last_name, full_name, email, phone_number, city, signup_date, last_updated_timestamp, scd2_hash,start_date, end_date, is_current, _processed_at
    ) VALUES (
        source.customer_id, source.first_name, source.last_name, source.full_name, source.email, source.phone_number, source.city, source.signup_date, source.last_updated_timestamp, source.source_hash, current_timestamp(), null, true, source._processed_at
    );